## Texas Super Tuesday

In [539]:
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz
import subprocess
import json

In [540]:
roddy = pd.read_csv('raw_data/roddyoriginals.csv')

roddy.to_csv("export_data/roddy_import.csv", index=False)

In [541]:
#Create full address for record name

roddy["Last Name"] = (
    roddy["Street Number"].astype("string").str.replace(r"\.0$", "", regex=True) + " " +
    roddy["Address"].str.title() + ", " +
    roddy["City Name"].str.title() + ", TX " +
    roddy["Zip"].astype(str).str.replace(r"\.0$", "", regex=True)
)

In [542]:
#Find County

texascounties = pd.read_csv('sf_data/texascounties.csv')

roddy['County'] = roddy['County Code'].map(
    texascounties.drop_duplicates(subset='County Code')
    .set_index('County Code')['County']
)

In [543]:
#Find Market

roddy['Market'] = roddy['County Code'].map(
    texascounties.drop_duplicates(subset='County Code')
    .set_index('County Code')['Market']
)

In [544]:
#Create unique identifier

market_map = {
    'TX Small County': 'TXSMALLCOUNTY',
    'Austin': 'AUSTIN',
    'Waco': 'WACO',
    'Dallas': 'DAL',
    'San Antonio': 'SAN_ANT',
    'Houston': 'HOUS'
}

roddy['APN'] = (
    roddy['Market'].map(market_map)
    + '_' + roddy['Street Number'].astype('string').str.replace(r'\.0$', '', regex=True)
    + '_' + roddy['Address'].astype('string').str.upper().str.replace(' ', '_', regex=False)
    + '_' + roddy['Zip'].astype('string').str.replace(r'\.0$', '', regex=True)
)

In [545]:
#Format County

roddy['County'] = roddy['County'].apply(
    lambda x: x + ' - TX' if pd.notna(x) and x != '' else ''
)

In [546]:
#Add Auction Status

roddy['Auction Status'] = roddy['Class'].apply(
    lambda x: 'Pending' if x == 'Single Family Residential' else 'Killed'
)

In [547]:
#Format Type of Property

roddy['Type of Property'] = roddy['Class'].replace({
    'Single Family Residential': 'SFR'
})

In [548]:
#Format Beds and Baths

roddy['Beds'] = roddy['B/B/G'].astype('string').str.split('/').str[0]
roddy['Baths'] = roddy['B/B/G'].astype('string').str.split('/').str[1]

In [549]:
#Format Pool

roddy = roddy.rename(columns={'Pool': 'notPool'})

roddy['Pool'] = roddy['notPool'].apply(
    lambda x: 'Yes' if x == 'P' else 'No'
)

In [550]:
#Format Owner

roddy['Property Owner Name'] = (
    roddy['Mortgagor First Name'].fillna('').astype(str).str.strip()
    + ' ' +
    roddy['Mortgagor Last Name'].fillna('').astype(str).str.strip()
).str.strip()

In [551]:
#Add Auction Date

roddy['Auction Date Updated'] = '11/3/2026'

In [552]:
#Add Auction Trustee Lookup

roddy['Auction Trustee (Lookup)'] = 'Lookup Needed'

In [553]:
#Add State

roddy['State'] = 'Texas'

In [554]:
#Rename Columns

roddy = roddy.rename(columns={
    'Report ID': 'Roddy Report ID',
    'Loan Type': 'Foreclosing Instrument',
    'Property Sq Footage': 'Bldg Sq Ft',
    'Legal Description 1': 'Legal Description',
    'Mortgagee': 'Beneficiary',
    'Trustee': 'Auction Trustee',
    'Volume': 'Foreclosing Instrument No.',
    'Original Loan Amount': 'Mortgage $',
    'Year Of Construction': 'Year Built',
    'Assessed Value': 'RPR',
    'Status': 'notStatus'
})

In [555]:
roddy[['County', 'Market', 'Last Name', 'APN']].head()

,County,Market,Last Name,APN
0,Dallas - TX,Dallas,"15375 Addison, Addison, TX 75001",DAL_15375_ADDISON_75001
1,Dallas - TX,Dallas,"4821 Carolyn Jean Dr, Balch Springs, TX 75180",DAL_4821_CAROLYN_JEAN_DR_75180
2,Dallas - TX,Dallas,"3821 Meredith Ln, Balch Springs, TX 75180",DAL_3821_MEREDITH_LN_75180
3,Dallas - TX,Dallas,"14853 Bell Manor Ct, Balch Springs, TX 75180",DAL_14853_BELL_MANOR_CT_75180
4,Dallas - TX,Dallas,"14914 Green Valley Dr, Balch Springs, TX 75180",DAL_14914_GREEN_VALLEY_DR_75180


## Match Addresses

In [556]:
query = """
SELECT
    Id,
    Name,
    Auction_Status__c,
    Auction_Trustee_Lookup__r.Name,
    Auction_Listing_ID__c,
    Online_Bidding__c,
    Auction_Opening_Bid__c
FROM Lead
WHERE (
    Auction_Date__c = LAST_N_DAYS:60
    OR Auction_Date__c = NEXT_N_DAYS:30)
AND State_Lookup__r.Name = 'Texas'
"""

result = subprocess.run(
    [
        "sf", "data", "query",
        "--query", query,
        "--target-org", "production_org",
        "--json"
    ],
    capture_output=True,
    text=True,
    env={**os.environ, "NO_COLOR": "1", "FORCE_COLOR": "0"}
)

if result.returncode != 0:
    print("Salesforce Error:")
    print(result.stderr)
    print(result.stdout[:1000])

else:
    # Remove ANSI color codes if any remain
    import re

    clean_output = re.sub(
        r'\x1b\[[0-9;]*m',
        '',
        result.stdout
    )

    data = json.loads(clean_output)

    sf_leads = pd.json_normalize(data['result']['records'])

    print(f"Successfully retrieved {len(sf_leads)} Salesforce records")
    display(sf_leads.head())

Successfully retrieved 5437 Salesforce records


,Id,Name,Auction_Status__c,Auction_Listing_ID__c,Online_Bidding__c,Auction_Opening_Bid__c,attributes.type,attributes.url,Auction_Trustee_Lookup__r.attributes.type,Auction_Trustee_Lookup__r.attributes.url,Auction_Trustee_Lookup__r.Name
0,00QQg00000h8lfOMAQ,"8048 Caladium Dr, North Richland Hills, TX 76180",Cancelled,2129172,Yes,160000.0,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000h...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Auction.com
1,00QQg00000h8lfYMAQ,"5667 Westhaven Dr, Fort Worth, TX 76132",Pending,None,None,NaN,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000h...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Lookup Needed
2,00QQg00000h8lfiMAA,"8364 Meadow Sweet Ln, Fort Worth, TX 76123",Active,None,None,264120.0,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000h...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Lookup Needed
3,00QQg00000h946xMAA,"3211 Kellie St, Sachse, TX 75048",Cancelled,None,No,NaN,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000h...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Xome.com
4,00QQg00000h9476MAA,"7218 Bursey Rd, North Richland Hills, TX 76182",Pending,None,None,NaN,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000h...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Lookup Needed


In [557]:
#Rename sf columns 

sf_leads = sf_leads.rename(columns={
    'Name': 'Last Name',
    'Auction_Status__c': 'Auction Status',
    'Auction_Trustee_Lookup__r.Name': 'Auction Trustee',
    'Auction_Listing_ID__c': 'Auction Listing ID',
    'Auction_Opening_Bid__c': 'Auction Opening Bid',
    'Online_Bidding__c': 'Online Bidding'
})

In [558]:
sf_leads[['Last Name', 'Auction Status', 'Auction Trustee',
          'Auction Listing ID', 'Auction Opening Bid']].head()

,Last Name,Auction Status,Auction Trustee,Auction Listing ID,Auction Opening Bid
0,"8048 Caladium Dr, North Richland Hills, TX 76180",Cancelled,Auction.com,2129172,160000.0
1,"5667 Westhaven Dr, Fort Worth, TX 76132",Pending,Lookup Needed,None,NaN
2,"8364 Meadow Sweet Ln, Fort Worth, TX 76123",Active,Lookup Needed,None,264120.0
3,"3211 Kellie St, Sachse, TX 75048",Cancelled,Xome.com,None,NaN
4,"7218 Bursey Rd, North Richland Hills, TX 76182",Pending,Lookup Needed,None,NaN


In [559]:
#Fuzzy Match

threshold = 85

# Normalize names
def normalize_name(value):
    if pd.isna(value):
        return ""
    return " ".join(str(value).upper().strip().split())

# Salesforce lookup
sf_lookup = {}

for name in sf_leads['Last Name'].dropna().unique():
    normalized = normalize_name(name)
    if normalized:
        sf_lookup.setdefault(normalized, name)

sf_names = list(sf_lookup.keys())

# Keep original Roddy Last Name
roddy['Roddy Last Name'] = roddy['Last Name']

# Fuzzy match
def fuzzy_match_name(name):
    normalized = normalize_name(name)

    if not normalized or not sf_names:
        return pd.Series([None, None])

    match = process.extractOne(
        normalized,
        sf_names,
        scorer=fuzz.ratio
    )

    if match:
        matched_name, score, _ = match
        return pd.Series([sf_lookup[matched_name], round(score, 2)])

    return pd.Series([None, None])

# Store Salesforce match and confidence
roddy[['Salesforce Last Name', 'Match Confidence']] = (
    roddy['Roddy Last Name'].apply(fuzzy_match_name)
)

# Use Salesforce Last Name only if confidence >= 85
roddy['Last Name'] = roddy['Roddy Last Name']

matched = roddy['Match Confidence'] >= threshold

roddy.loc[matched, 'Last Name'] = (
    roddy.loc[matched, 'Salesforce Last Name']
)

In [560]:
#View matches

display(roddy[
    ['Roddy Last Name', 'Salesforce Last Name', 'Match Confidence']
])

,Roddy Last Name,Salesforce Last Name,Match Confidence
0,"15375 Addison, Addison, TX 75001","1505 W Johnson St, Denison, TX 75020",67.65
1,"4821 Carolyn Jean Dr, Balch Springs, TX 75180","14825 Magnolia Ln, Balch Springs, TX 75180",78.16
2,"3821 Meredith Ln, Balch Springs, TX 75180","14825 Magnolia Ln, Balch Springs, TX 75180",79.52
3,"14853 Bell Manor Ct, Balch Springs, TX 75180","14825 Magnolia Ln, Balch Springs, TX 75180",81.40
4,"14914 Green Valley Dr, Balch Springs, TX 75180","11816 Sunrise Dr, Balch Springs, TX 75180",75.86
...,...,...,...
2419,"19517 Stage Line Trl, Pflugerville, TX 78660","13904 Stripling Ln, Pflugerville, TX 78660",79.07
2420,"15007 Chamberlain Ct, Reno, TX 78724","416 Cumberland Ct, Arlington, TX 76014",67.57
2421,"17308 Monastrell Ln 43, Round Rock, TX 78664","1507 Balmorhea Ln, Round Rock, TX 78664",77.11
2422,"22111 Briarcliff Dr, Spicewood, TX 78669","1516 Likeness Rd, Spicewood, TX 78669",72.73


In [561]:
#Look at differences

#Common street suffixes
suffix_map = {
    'DRIVE': 'DR',
    'STREET': 'ST',
    'AVENUE': 'AVE',
    'ROAD': 'RD',
    'LANE': 'LN',
    'BOULEVARD': 'BLVD',
    'COURT': 'CT',
    'CIRCLE': 'CIR',
    'PLACE': 'PL',
    'TERRACE': 'TER',
    'TRAIL': 'TRL',
    'PARKWAY': 'PKWY',
    'HIGHWAY': 'HWY',
    'WAY': 'WAY'
}

def get_address_parts(address):
    if pd.isna(address):
        return '', ''

    address = str(address).strip()
    parts = [p.strip() for p in address.split(',')]

    # Expected format: Street, City, TX ZIP
    street = parts[0] if len(parts) >= 1 else ''
    city = parts[1] if len(parts) >= 3 else ''

    return street.upper(), city.upper()

def normalize_street(street):
    street = re.sub(r'[^\w\s]', '', street.upper())
    words = street.split()
    return ' '.join(suffix_map.get(word, word) for word in words)

def compare_addresses(row):
    roddy_street, roddy_city = get_address_parts(row['Roddy Last Name'])
    sf_street, sf_city = get_address_parts(row['Salesforce Last Name'])

    # Detect abbreviation-only differences
    suffix_difference = (
        roddy_street != sf_street
        and normalize_street(roddy_street) == normalize_street(sf_street)
    )

    city_difference = roddy_city != sf_city

    return pd.Series({
        'Street Suffix Difference': suffix_difference,
        'City Difference': city_difference,
        'Roddy City': roddy_city,
        'Salesforce City': sf_city
    })

# Compare matched records
address_comparison = roddy[
    (roddy['Match Confidence'] >= 85) &
    roddy['Salesforce Last Name'].notna()
].copy()

address_comparison = address_comparison.join(
    address_comparison.apply(compare_addresses, axis=1)
)

In [562]:
differences = address_comparison[
    address_comparison['Street Suffix Difference'] |
    address_comparison['City Difference']
]

display(differences[
    [
        'Roddy Last Name',
        'Salesforce Last Name',
        'Match Confidence',
        'Street Suffix Difference',
        'City Difference'
    ]
])

,Roddy Last Name,Salesforce Last Name,Match Confidence,Street Suffix Difference,City Difference
139,"10606 Shady Trl Ste 20, Dallas, TX 75220","10606 Shady Trl, Ste 20, Dallas, TX 75220",98.77,False,True
168,"601 Belclaire Terrace, Desoto, TX 75115","601 Belclaire Ter, Desoto, TX 75115",94.59,True,False
634,"536 Graceful Pl, Lavon, TX 75166","536 Graceful Place, Lavon, TX 75166",95.52,True,False
989,"8031 Swindow Cir, Converse, TX 78109","8031 Swindow Circle, Converse, TX 78109",96.00,True,False
1169,"15139 Longtailed Duck, San Antonio, TX 78253","15139 Longtailed Duck, San Antonio, TX 78253",100.00,True,False
1183,"6011 Akin Place, San Antonio, TX 78261","6011 Akin Pl, San Antonio, TX 78261",95.89,True,False
1212,"3707 William Penn Dr, San Antonio, TX 78230","3707 William Penn Drive, San Antonio, TX 78230",96.63,True,False
1233,"342 Haverford Dr, San Antonio, TX 78217","342 Haverford Drive, San Antonio, TX 78217",96.30,True,False
1255,"538 Crestway Dr, San Antonio, TX 78239","538 Crestway Drive, San Antonio, TX 78239",96.20,True,False
1878,"7139 Palmetto St, Houston, TX 77087","7139 Palmetto Street, Houston, TX 77087",94.59,True,False


In [563]:
print("Street suffix differences:",
      address_comparison['Street Suffix Difference'].sum())

print("City differences:",
      address_comparison['City Difference'].sum())

print("Both:",
      (
          address_comparison['Street Suffix Difference'] &
          address_comparison['City Difference']
      ).sum())

Street suffix differences: 10
City differences: 2
Both: 0


In [564]:
low_confidence = roddy[
    (roddy['Match Confidence'] < 85) |
    (roddy['Match Confidence'].isna())
]

display(low_confidence[
    ['Last Name', 'Match Confidence']
])

,Last Name,Match Confidence
0,"15375 Addison, Addison, TX 75001",67.65
1,"4821 Carolyn Jean Dr, Balch Springs, TX 75180",78.16
2,"3821 Meredith Ln, Balch Springs, TX 75180",79.52
3,"14853 Bell Manor Ct, Balch Springs, TX 75180",81.40
4,"14914 Green Valley Dr, Balch Springs, TX 75180",75.86
...,...,...
2419,"19517 Stage Line Trl, Pflugerville, TX 78660",79.07
2420,"15007 Chamberlain Ct, Reno, TX 78724",67.57
2421,"17308 Monastrell Ln 43, Round Rock, TX 78664",77.11
2422,"22111 Briarcliff Dr, Spicewood, TX 78669",72.73


In [565]:
# Count Salesforce Matches
matched = roddy['Match Confidence'].ge(85).sum()
not_matched = roddy['Match Confidence'].lt(85).sum() + roddy['Match Confidence'].isna().sum()

print("Total Properties:", len(roddy))
print("Matched to Salesforce:", matched)
print("Not Matched to Salesforce:", not_matched)
print("Match Rate:", round(matched / len(roddy) * 100, 2) if len(roddy) else 0, "%")

Total Properties: 2424
Matched to Salesforce: 610
Not Matched to Salesforce: 1814
Match Rate: 25.17 %


In [566]:
# Clear Salesforce data for matches below 85%
salesforce_columns = [
    'Salesforce Last Name',
    'Salesforce Auction Trustee',
    'Salesforce Online Bidding'
]

roddy.loc[
    roddy['Match Confidence'].lt(85) |
    roddy['Match Confidence'].isna(),
    salesforce_columns
] = ''

In [567]:
# Count blank Salesforce Last Names
blank_count = (
    roddy['Salesforce Last Name'].isna() |
    roddy['Salesforce Last Name'].astype(str).str.strip().eq('')
).sum()

print("Blank Salesforce Last Names:", blank_count)

Blank Salesforce Last Names: 1814


## Lead Manager

In [568]:
# Create Salesforce lookup using matched Last Name
sf_lookup = sf_leads.set_index('Last Name')

# Get Salesforce Auction Trustee
roddy['Salesforce Auction Trustee'] = (
    roddy['Salesforce Last Name']
    .map(sf_lookup['Auction Trustee'])
)

# Get Salesforce Online Bidding
roddy['Salesforce Online Bidding'] = (
    roddy['Salesforce Last Name']
    .map(sf_lookup['Online Bidding'])
)

In [569]:
leadmanagers = pd.read_csv('sf_data/leadmanagers.csv')

# County lookup
county_lookup = (
    texascounties.drop_duplicates(subset='County Code')
    .set_index('County Code')
)

# Get both managers
roddy['Online Manager'] = roddy['County Code'].map(
    county_lookup['Online Lead Manager']
)

roddy['In Person Manager'] = roddy['County Code'].map(
    county_lookup['In Person Lead Manager']
)

# Use online manager if Salesforce trustee is Auction.com
# OR Salesforce Online Bidding is Yes
use_online = (
    roddy['Salesforce Auction Trustee'].eq('Auction.com')
    |
    roddy['Salesforce Online Bidding'].eq('Yes')
)

# Assign Lead Manager Name
roddy['Lead Manager Name'] = np.where(
    use_online,
    roddy['Online Manager'],
    roddy['In Person Manager']
)

# Find Salesforce ID
manager_lookup = (
    leadmanagers.drop_duplicates(subset='Lead Manager')
    .set_index('Lead Manager')['Salesforce ID']
)

roddy['Lead Manager'] = (
    roddy['Lead Manager Name'].map(manager_lookup)
)

# Remove temporary columns
roddy = roddy.drop(
    columns=['Online Manager', 'In Person Manager']
)

In [570]:
display(roddy[
    [
        'Last Name',
        'Salesforce Last Name',
        'Salesforce Auction Trustee',
        'Salesforce Online Bidding',
        'Lead Manager Name',
        'Lead Manager'
    ]
].head(20))

,Last Name,Salesforce Last Name,Salesforce Auction Trustee,Salesforce Online Bidding,Lead Manager Name,Lead Manager
0,"15375 Addison, Addison, TX 75001",,NaN,NaN,Brian Glasser,005Qg00000V0o9R
1,"4821 Carolyn Jean Dr, Balch Springs, TX 75180",,NaN,NaN,Brian Glasser,005Qg00000V0o9R
2,"3821 Meredith Ln, Balch Springs, TX 75180",,NaN,NaN,Brian Glasser,005Qg00000V0o9R
3,"14853 Bell Manor Ct, Balch Springs, TX 75180",,NaN,NaN,Brian Glasser,005Qg00000V0o9R
4,"14914 Green Valley Dr, Balch Springs, TX 75180",,NaN,NaN,Brian Glasser,005Qg00000V0o9R
5,"14601 Cimarron Dr, Balch Springs, TX 75180",,NaN,NaN,Brian Glasser,005Qg00000V0o9R
6,"14306 Horseshoe Trl, Balch Springs, TX 75180",,NaN,NaN,Brian Glasser,005Qg00000V0o9R
7,"11816 Sunrise Dr, Balch Springs, TX 75180","11816 Sunrise Dr, Balch Springs, TX 75180",Unknown,No,Brian Glasser,005Qg00000V0o9R
8,"2706 Glenwood St, Carrollton, TX 75006",,NaN,NaN,Brian Glasser,005Qg00000V0o9R
9,"2706 Glenwood Ct, Carrollton, TX 75006",,NaN,NaN,Brian Glasser,005Qg00000V0o9R


In [571]:
#No lead manager

display(roddy[
    roddy['Lead Manager Name'].isna() |
    (roddy['Lead Manager Name'].astype(str).str.strip() == '')
])

,Upload Time,Roddy Report ID,Estimated Equity,Estimated Margin percent,Est Unpaid Bal,Clerk Number,Year of Report,Month of Report,City Name,County Code,...,Auction Date Updated,Auction Trustee (Lookup),State,Roddy Last Name,Salesforce Last Name,Match Confidence,Salesforce Auction Trustee,Salesforce Online Bidding,Lead Manager Name,Lead Manager
323,2026-08-26 12:37:33,5,104276.0,36.0,185471,5,2026,11,ARLINGTON,TA,...,11/3/2026,Lookup Needed,Texas,"801 Dove Meadows Dr, Arlington, TX 76002",,76.92,NaN,NaN,NaN,NaN
324,2026-08-26 12:44:33,6,221592.0,68.0,104035,6,2026,11,ARLINGTON,TA,...,11/3/2026,Lookup Needed,Texas,"1104 Port Au Prince Dr, Arlington, TX 76012",,74.70,NaN,NaN,NaN,NaN
325,2026-08-26 14:03:38,16,192517.0,74.0,67291,16,2026,11,ARLINGTON,TA,...,11/3/2026,Lookup Needed,Texas,"7607 Kittery Ln, Arlington, TX 76002",,78.26,NaN,NaN,NaN,NaN
326,2026-08-26 14:41:22,22,308665.0,78.0,88582,22,2026,11,ARLINGTON,TA,...,11/3/2026,Lookup Needed,Texas,"6621 Fairglen Dr, Arlington, TX 76002",,80.00,NaN,NaN,NaN,NaN
327,2026-08-30 00:38:50,28,-53484.0,-16.0,395163,28,2026,11,ARLINGTON,TA,...,11/3/2026,Lookup Needed,Texas,"2412 Cross Timbers Tr, Arlington, TX 76006",,77.11,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2296,2026-10-13 12:27:05,97,104179.0,45.0,127362,94,2026,11,HORIZON CITY,EP,...,11/3/2026,Lookup Needed,Texas,"13621 Lancashire St, Horizon City, TX 79928",,75.00,NaN,NaN,NaN,NaN
2297,2026-10-13 12:38:18,99,130284.0,56.0,104107,106,2026,11,HORIZON CITY,EP,...,11/3/2026,Lookup Needed,Texas,"19005 Armington Dr, Horizon City, TX 79928","17011 Darrington Rd, Horizon City, TX 79928",87.06,Servicelinkauction.com,None,NaN,NaN
2298,2026-09-05 05:07:05,16,10241.0,5.0,199549,41,2026,11,SOCORRO,EP,...,11/3/2026,Lookup Needed,Texas,"709 Villas Del Este, Socorro, TX 79927",,70.13,NaN,NaN,NaN,NaN
2299,2026-09-05 05:21:37,18,49174.0,19.0,203253,44,2026,11,SOCORRO,EP,...,11/3/2026,Lookup Needed,Texas,"11533 Leonor Duran, Socorro, TX 79927",,69.57,NaN,NaN,NaN,NaN


In [572]:
# Add Killed Reason
roddy['Killed Reason'] = roddy['Auction Status'].apply(
    lambda x: 'Property Type' if x == 'Killed' else ''
)

In [573]:
# Update Auction Trustee Lookup
roddy.loc[
    roddy['Salesforce Auction Trustee'] == 'Auction.com',
    'Auction Trustee (Lookup)'
] = 'Auction.com'

In [574]:
roddy.to_csv("export_data/roddy_import.csv", index=False)